# Sistem Prediksi Nilai Tukar Rupiah (USD/IDR) Berbasis Machine Learning
### dengan Dukungan Analitik Makroekonomi untuk Pengambilan Keputusan Bisnis

**Proyek Tugas Besar - Desain Analisis Sains Data (DASD) x Sistem Informasi Analitik Bisnis (SIAB)**

---

## Ringkasan Eksekutif (baca ini dulu)

Notebook ini membangun sistem prediksi nilai tukar USD/IDR. **Satu hal yang harus dipahami
sejak awal:** nilai tukar harian bergerak sangat mendekati *random walk* (acak), sehingga
ada **dua pertanyaan berbeda** dengan tingkat kesulitan yang sangat berbeda:

| Pertanyaan | Bisa dijawab? | Metrik |
|---|---|---|
| "Berapa **angka persis** kurs besok?" | **Sulit** - model hanya unggul **tipis** atas tebakan "besok = hari ini" | MAE, RMSE, MAPE |
| "Kurs besok **naik atau turun**?" | **Bisa** - model jelas lebih baik dari tebakan koin (~65% vs 50%) | Akurasi Arah |

Karena keunggulan pada angka persis hanya tipis, kita **memilih model juara berdasarkan Akurasi
Arah** - metrik yang keunggulannya paling jelas, dapat dimenangkan, dan paling berguna untuk
keputusan treasury (kapan membeli/menjual valas).

### Keputusan desain utama
1. **Target = return harian (log-return)**, bukan level harga. Return bersifat *stasioner*
   sehingga selalu berada di rentang yang dikenal model (menghindari kegagalan ekstrapolasi
   pohon pada data level yang terus naik).
2. **Semua fitur stasioner** (perubahan/return), bebas *look-ahead bias*.
3. **Baseline Naif (persistence)** sebagai tolok ukur wajib.
4. **Tiga model dibandingkan**: Linear Regression (linear), Random Forest (bagging),
   XGBoost (boosting) - mewakili tiga keluarga algoritma utama untuk data tabular.
5. **Juara dipilih otomatis berdasarkan Akurasi Arah**, lalu dipakai untuk forecasting & aplikasi.
6. **Reproducible:** seluruh proses acak memakai `RANDOM_STATE = 42`.

## 1. Import Library

Memuat pustaka untuk manipulasi data, visualisasi, pemodelan, dan penyimpanan model.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
)
from xgboost import XGBRegressor
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["figure.dpi"] = 100

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Library siap. RANDOM_STATE =", RANDOM_STATE)

## 2. Deskripsi Dataset

Bagian ini menjawab ketentuan tugas (komponen deskripsi dataset).

| Komponen | Keterangan |
|---|---|
| **Nama dataset** | Daily Multi-Asset Macroeconomic and Financial Time Series Dataset (Indonesia & Global Markets) |
| **Domain** | Keuangan / Makroekonomi |
| **Sumber** | Dataset publik (kompilasi harga pasar & indikator makro harian) |
| **Jumlah data** | ~4.293 baris (hari kerja, 2010 - 2026) |
| **Jumlah fitur** | 9 variabel + 1 kolom tanggal |
| **Variabel target** | **USDIDR** (diolah menjadi *log-return* harian) |
| **Jenis data** | Numerik + deret waktu (time series) |
| **Alasan pemilihan** | USD/IDR adalah variabel kunci bagi importir/eksportir & treasury; indikator makro (minyak, emas, suku bunga, VIX) adalah penggerak fundamentalnya |
| **Potensi risiko data** | Kesalahan input (typo nilai), missing value pada hari libur, ketidakseimbangan rezim pasar (krisis vs normal) |

**Daftar variabel:**
- `USDIDR` - nilai tukar USD ke Rupiah (**target**)
- `OIL` - harga minyak mentah (WTI), USD/barel
- `GOLD` - harga emas, USD/oz
- `SP500` - indeks saham AS
- `IHSG` - indeks saham Indonesia
- `VIX` - indeks volatilitas/"ketakutan" pasar global
- `CPI` - inflasi Indonesia (%)
- `BI_rate` - suku bunga acuan Bank Indonesia (%)
- `US_rate` - suku bunga acuan The Fed (%)

## 3. Load Data dan Eksplorasi (EDA)

Memahami struktur, kualitas, dan karakteristik data sebelum pemodelan.

In [ ]:
# Loader: cari beberapa kemungkinan nama file
CSV_CANDIDATES = [
    "Daily Multi-Asset Macroeconomic and Financial Time Series Dataset (IndonesiaGlobal Markets).csv",
    "Daily_Multi-Asset_Macroeconomic_and_Financial_Time_Series_Dataset__IndonesiaGlobal_Markets_.csv",
]
df = None
for path in CSV_CANDIDATES:
    if os.path.exists(path):
        df = pd.read_csv(path, parse_dates=["Date"])
        print(f"Dataset '{path}' berhasil dimuat.")
        break
if df is None:
    raise FileNotFoundError("Dataset tidak ditemukan. Letakkan file CSV di folder yang sama.")

df = df.sort_values("Date").reset_index(drop=True)
kolom_numerik = df.select_dtypes(include=[np.number]).columns.tolist()
print("Dimensi data:", df.shape)

In [ ]:
print("=== TIPE DATA ==="); print(df.dtypes)
print("\n=== 5 BARIS AWAL ==="); display(df.head())
print("=== 5 BARIS AKHIR ==="); display(df.tail())
print("=== STATISTIK DESKRIPTIF ==="); display(df.describe().T)

In [ ]:
# 3.a Cek missing value
missing = df.isnull().sum()
ringkasan_missing = pd.DataFrame({
    "jumlah_missing": missing,
    "persen_missing(%)": (missing / len(df) * 100).round(2)
})
print("=== MISSING VALUE ==="); display(ringkasan_missing)

plt.figure(figsize=(12, 5))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis", yticklabels=False)
plt.title("Heatmap Missing Values (kuning = nilai hilang)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_heatmap_missing_values.png"), bbox_inches="tight")
plt.show()

In [ ]:
# 3.b Cek data duplikat (ketentuan preprocessing)
dup_penuh = df.duplicated().sum()
dup_tanggal = df.duplicated(subset=["Date"]).sum()
print(f"Jumlah baris duplikat penuh : {dup_penuh}")
print(f"Jumlah tanggal duplikat     : {dup_tanggal}")
if dup_penuh == 0 and dup_tanggal == 0:
    print(">> Tidak ada duplikat. Tidak perlu penghapusan.")

In [ ]:
# 3.c Deteksi OUTLIER / kesalahan input pada USDIDR
# Strategi: cari "spike" satu hari -- nilai yang jauh lebih kecil dari hari sebelum DAN sesudahnya.
# Ini menandai kesalahan input (digit hilang), BUKAN pergerakan pasar nyata.
prev = df["USDIDR"].shift(1)
nxt  = df["USDIDR"].shift(-1)
mask_error = (df["USDIDR"] < 0.6 * prev) & (df["USDIDR"] < 0.6 * nxt)
print("=== DUGAAN KESALAHAN INPUT pada USDIDR ===")
display(df.loc[mask_error, ["Date", "USDIDR"]])

# Boxplot beberapa variabel kunci untuk gambaran sebaran/outlier
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, ["USDIDR", "OIL", "VIX"]):
    sns.boxplot(y=df[col], ax=ax, color="#4C72B0")
    ax.set_title(f"Boxplot {col}")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "02_boxplot_outlier.png"), bbox_inches="tight")
plt.show()
print("Catatan: OIL pernah negatif (-37.6 pada 20 Apr 2020) -- ini PERISTIWA NYATA (krisis harga")
print("minyak WTI), jadi DIPERTAHANKAN. Yang dikoreksi hanya kesalahan input USDIDR di atas.")

In [ ]:
# 3.d Tren USDIDR + korelasi antar fitur (level)
plt.figure(figsize=(14, 6))
plt.plot(df["Date"], df["USDIDR"], color="darkred", linewidth=1)
plt.title("Tren Nilai Tukar USD/IDR (2010 - 2026)")
plt.xlabel("Tahun"); plt.ylabel("USD/IDR")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "03_tren_usdidr.png"), bbox_inches="tight")
plt.show()

plt.figure(figsize=(11, 9))
sns.heatmap(df[kolom_numerik].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Heatmap Korelasi Antar Fitur (level)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_heatmap_korelasi.png"), bbox_inches="tight")
plt.show()

**Interpretasi EDA:**
- Tren jangka panjang menunjukkan **pelemahan Rupiah** dengan lonjakan pada periode krisis.
- Korelasi antar **level** tinggi karena banyak deret bersifat tren (non-stasioner). Ini sinyal
  bahwa pemodelan sebaiknya pada **perubahan harian (return)**, bukan level, agar relasi valid
  dan model tidak gagal ekstrapolasi.
- Ditemukan **2 kesalahan input pada USDIDR** (nilai ~888 & ~892 yang seharusnya ~8.900) yang
  akan dikoreksi pada tahap preprocessing agar tidak menciptakan return palsu yang ekstrem.

## 4. Preprocessing Data (Pendekatan Return)

**Tujuan:** membangun representasi data yang **stasioner** dan **bebas look-ahead bias**.
- **Target** = log-return USD/IDR hari ini: `r_t = ln(USDIDR_t / USDIDR_{t-1})`.
- **Fitur** = hanya informasi yang tersedia sampai kemarin (`t-1`): lag return, statistik
  rolling return (momentum & volatilitas), perubahan indikator makro, dan selisih suku bunga.

Rekonstruksi level saat evaluasi: `USDIDR_t = USDIDR_{t-1} * exp(r_hat_t)` (one-step-ahead yang adil).

In [ ]:
# 4.1 Koreksi kesalahan input + bersihkan missing value
df_clean = df.copy()

# Koreksi outlier USDIDR: jadikan NaN lalu interpolasi dari tetangga (linear)
n_error = int(mask_error.sum())
df_clean.loc[mask_error, "USDIDR"] = np.nan
df_clean["USDIDR"] = df_clean["USDIDR"].interpolate(method="linear")
print(f"{n_error} kesalahan input USDIDR dikoreksi via interpolasi.")

# Set index waktu lalu isi missing (ffill->bfill)
df_clean["Date"] = pd.to_datetime(df_clean["Date"])
df_clean = df_clean.set_index("Date").sort_index()
df_clean[kolom_numerik] = df_clean[kolom_numerik].ffill().bfill()
print("Missing tersisa:", int(df_clean.isnull().sum().sum()))
print("Rentang waktu :", df_clean.index.min().date(), "s.d.", df_clean.index.max().date())

In [ ]:
# 4.2 Feature engineering berbasis return (stasioner & bebas look-ahead)
d = df_clean.copy()

# Target: log-return USD/IDR hari ini
d["ret"] = np.log(d["USDIDR"]).diff()
# Harga kemarin (untuk rekonstruksi level; BUKAN fitur)
d["USDIDR_prev"] = d["USDIDR"].shift(1)

# Lag return (momentum) -- info sampai t-1
for l in [1, 2, 3, 5, 10]:
    d[f"ret_lag{l}"] = d["ret"].shift(l)

# Statistik rolling return (di-shift 1 agar hanya memakai masa lalu)
d["ret_roll7_mean"]  = d["ret"].shift(1).rolling(7).mean()
d["ret_roll7_std"]   = d["ret"].shift(1).rolling(7).std()
d["ret_roll30_mean"] = d["ret"].shift(1).rolling(30).mean()
d["ret_roll30_std"]  = d["ret"].shift(1).rolling(30).std()

# Perubahan indikator makro (return), pakai nilai kemarin (shift 1)
makro_pct = ["OIL", "GOLD", "SP500", "IHSG", "VIX", "CPI"]
for col in makro_pct:
    d[f"{col}_chg_lag1"] = d[col].pct_change().shift(1)

# Selisih suku bunga (interest rate differential)
d["rate_diff"] = d["BI_rate"] - d["US_rate"]
d["rate_diff_lag1"] = d["rate_diff"].shift(1)
d["rate_diff_chg_lag1"] = d["rate_diff"].diff().shift(1)

# Fitur waktu (diketahui di muka)
d["day_of_week"] = d.index.dayofweek
d["month"] = d.index.month
print("Fitur turunan dibuat. Total kolom:", d.shape[1])

In [ ]:
# 4.3 Daftar fitur (X) dan target (y)
FEATURE_COLS = (
    [f"ret_lag{l}" for l in [1, 2, 3, 5, 10]]
    + ["ret_roll7_mean", "ret_roll7_std", "ret_roll30_mean", "ret_roll30_std"]
    + [f"{c}_chg_lag1" for c in makro_pct]
    + ["rate_diff_lag1", "rate_diff_chg_lag1", "day_of_week", "month"]
)
TARGET_COL = "ret"

sebelum = len(d)
d_model = d.dropna(subset=FEATURE_COLS + [TARGET_COL, "USDIDR", "USDIDR_prev"]).copy()
print(f"Baris dibuang (akibat lag/rolling): {sebelum - len(d_model)} | Siap model: {len(d_model)}")
print(f"Jumlah fitur: {len(FEATURE_COLS)}")

In [ ]:
# 4.4 Split kronologis 80/20 (WAJIB kronologis untuk time series)
split_idx = int(len(d_model) * 0.8)
train, test = d_model.iloc[:split_idx], d_model.iloc[split_idx:]
X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET_COL]
print(f"Latih: {len(train)} baris ({train.index.min().date()} s.d. {train.index.max().date()})")
print(f"Uji  : {len(test)} baris ({test.index.min().date()} s.d. {test.index.max().date()})")

# 4.5 Normalisasi (fit HANYA pada train -> hindari kebocoran data)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("Normalisasi selesai.")

**Interpretasi preprocessing:** target kini berupa **return harian yang stasioner**, sehingga
tetap berada di rentang yang dikenal model pada periode uji (menghilangkan masalah ekstrapolasi).
Scaler hanya di-*fit* pada data latih agar tidak ada kebocoran informasi dari masa depan.

## 5. Pemilihan Algoritma: Kenapa Tiga Model Ini?

Masalah sudah dibingkai sebagai **regresi tabular bersupervisi** (target = return, fitur = lag
return + perubahan makro). Untuk format ini, kita membandingkan **tiga keluarga algoritma utama**:

| Model | Keluarga | Alasan dipilih |
|---|---|---|
| **Linear Regression** | Linear | Pembanding paling sederhana, cepat, interpretable. Jika model kompleks tak mengalahkannya, itu bukti relasi mendekati linear/acak. |
| **Random Forest** | Ensemble *bagging* | Menangkap relasi non-linear & interaksi fitur, tahan terhadap overfitting, memberi *feature importance*. |
| **XGBoost** | Ensemble *boosting* | Sering menjadi *state-of-the-art* untuk data tabular; membangun pohon secara bertahap mengoreksi error. |

**Mengapa bukan model lain?**
- **ARIMA / Prophet** - model deret *univariat* (hanya histori harga sendiri). Beda paradigma; tidak memanfaatkan fitur makro yang menjadi inti analisis ini.
- **LSTM / Deep Learning** - butuh data jauh lebih besar + tuning berat, sulit dijustifikasi untuk ~4.000 baris, dan berat untuk deployment ringan (cPanel).
- **KNN / SVM** - cenderung lebih lemah & lambat untuk jumlah fitur ini; tidak menambah nilai dibanding trio di atas.
- **Moving Average / Naif** - justru sudah dipakai sebagai **baseline** pembanding.

Ketiga model dilatih untuk memprediksi **return** (`ret`), lalu level direkonstruksi saat evaluasi.

## 6. Pemodelan Machine Learning

### Model 1: Random Forest Regression

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=300, max_depth=None, min_samples_leaf=5,
    n_jobs=-1, random_state=RANDOM_STATE,
)
rf_model.fit(X_train_scaled, y_train)
rf_ret = rf_model.predict(X_test_scaled)
print("Random Forest dilatih. Contoh prediksi return:", np.round(rf_ret[:3], 5))

### Model 2: XGBoost Regression

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=300, learning_rate=0.03, max_depth=4,
    subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
    random_state=RANDOM_STATE, n_jobs=-1,
)
xgb_model.fit(X_train_scaled, y_train)
xgb_ret = xgb_model.predict(X_test_scaled)
print("XGBoost dilatih. Contoh prediksi return:", np.round(xgb_ret[:3], 5))

### Model 3: Linear Regression

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_ret = lr_model.predict(X_test_scaled)
print("Linear Regression dilatih. Contoh prediksi return:", np.round(lr_ret[:3], 5))

## 7. Evaluasi Model

Penilaian **adil & jujur**. Metrik level (MAE/RMSE/MAPE/R^2 dari hasil rekonstruksi) +:
- **Baseline Naif (Persistence):** prediksi return = 0 ("besok = harga hari ini"). Tolok ukur wajib.
- **Akurasi Arah (Directional Accuracy):** % hari model menebak benar arah (naik/turun) -
  **inilah metrik penentu juara**, karena ini yang benar-benar bisa dimenangkan & paling
  bermakna untuk keputusan bisnis.

In [ ]:
# Rekonstruksi level: harga_hat_t = harga_kemarin * exp(return_hat_t)
prev_level   = test["USDIDR_prev"].values
actual_level = test["USDIDR"].values
actual_ret   = y_test.values

def level_dari_return(ret_pred):
    return prev_level * np.exp(ret_pred)

def metrik_level(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2   = r2_score(y_true, y_pred)
    return mae, rmse, mape, r2

def directional_accuracy(ret_true, ret_pred):
    mask = ret_true != 0
    return float((np.sign(ret_pred[mask]) == np.sign(ret_true[mask])).mean() * 100)

prediksi_return = {"Random Forest": rf_ret, "XGBoost": xgb_ret, "Linear Regression": lr_ret}

rows = {}
naive_level = level_dari_return(np.zeros_like(actual_ret))
mae, rmse, mape, r2 = metrik_level(actual_level, naive_level)
rows["Baseline Naif (Persistence)"] = [mae, rmse, mape, r2, np.nan]
for nama, ret_pred in prediksi_return.items():
    lvl = level_dari_return(ret_pred)
    mae, rmse, mape, r2 = metrik_level(actual_level, lvl)
    da = directional_accuracy(actual_ret, ret_pred)
    rows[nama] = [mae, rmse, mape, r2, da]

tabel_eval = pd.DataFrame(rows, index=["MAE", "RMSE", "MAPE(%)", "R2", "Dir.Acc(%)"]).T.round(4)
print("=== PERBANDINGAN KINERJA (pada LEVEL hasil rekonstruksi) ===")
display(tabel_eval)
tabel_eval.to_csv(os.path.join(OUTPUT_DIR, "evaluasi_model.csv"))

In [ ]:
# === PEMILIHAN JUARA: berdasarkan AKURASI ARAH tertinggi ===
model_ml = ["Random Forest", "XGBoost", "Linear Regression"]
nama_terbaik = tabel_eval.loc[model_ml, "Dir.Acc(%)"].idxmax()

print(f">> MODEL JUARA (akurasi arah tertinggi): {nama_terbaik}")
print(f"   Akurasi Arah : {tabel_eval.loc[nama_terbaik, 'Dir.Acc(%)']:.2f}%  (acak = 50%)")
rmse_juara = tabel_eval.loc[nama_terbaik, "RMSE"]
rmse_base  = tabel_eval.loc["Baseline Naif (Persistence)", "RMSE"]
print(f"   RMSE         : {rmse_juara:.2f}  (baseline naif = {rmse_base:.2f})")
print()
if rmse_juara < rmse_base:
    selisih = (1 - rmse_juara / rmse_base) * 100
    print(f"Catatan kejujuran: model juara mengalahkan baseline naif pada RMSE, namun hanya")
    print(f"unggul TIPIS ({selisih:.1f}% lebih kecil). Pada nilai tukar harian, keunggulan sekecil")
    print("ini wajar -- pasar mendekati efisien (random walk). Keunggulan yang paling jelas &")
    print("berguna justru ada pada AKURASI ARAH, sehingga itulah kriteria pemilihan juara.")
else:
    print("Catatan kejujuran: pada RMSE/MAE level, model belum mengalahkan baseline naif. Itu wajar")
    print("pada nilai tukar harian (pasar efisien). Juara dipilih dari AKURASI ARAH yang bermakna.")

MODEL_TERBAIK = {"Random Forest": rf_model, "XGBoost": xgb_model, "Linear Regression": lr_model}[nama_terbaik]
RET_TERBAIK = prediksi_return[nama_terbaik]

In [ ]:
# Bar chart RMSE & MAE (termasuk baseline)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
warna = ["#999999", "#4C72B0", "#55A868", "#C44E52"]
tabel_eval["RMSE"].plot(kind="bar", ax=axes[0], color=warna, edgecolor="black")
axes[0].set_title("Perbandingan RMSE (Rupiah) - termasuk Baseline")
axes[0].set_ylabel("RMSE"); axes[0].tick_params(axis="x", rotation=20)
tabel_eval["MAE"].plot(kind="bar", ax=axes[1], color=warna, edgecolor="black")
axes[1].set_title("Perbandingan MAE (Rupiah) - termasuk Baseline")
axes[1].set_ylabel("MAE"); axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_perbandingan_rmse_mae.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Akurasi arah (garis 50% = tebakan acak)
da_series = tabel_eval.loc[model_ml, "Dir.Acc(%)"]
plt.figure(figsize=(9, 5))
bars = da_series.plot(kind="bar", color=["#55A868" if m==nama_terbaik else "#4C72B0" for m in model_ml], edgecolor="black")
plt.axhline(50, color="red", linestyle="--", label="Tebakan acak (50%)")
plt.title("Akurasi Arah Pergerakan (Directional Accuracy) - hijau = juara")
plt.ylabel("Akurasi (%)"); plt.xticks(rotation=15); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_directional_accuracy.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Prediksi vs aktual (level) untuk model juara
level_terbaik = level_dari_return(RET_TERBAIK)
plt.figure(figsize=(15, 6))
plt.plot(test.index, actual_level, label="Aktual", color="black", linewidth=1.5)
plt.plot(test.index, level_terbaik, label=f"Prediksi ({nama_terbaik})", color="orange", alpha=0.85)
plt.plot(test.index, naive_level, label="Baseline Naif", color="gray", alpha=0.5, linestyle="--")
plt.title(f"Prediksi vs Aktual USD/IDR (Level) - {nama_terbaik}")
plt.xlabel("Tanggal"); plt.ylabel("USD/IDR"); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "07_prediksi_vs_aktual.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Distribusi residual model juara
residual = actual_level - level_terbaik
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].hist(residual, bins=50, color="purple", edgecolor="white")
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"Distribusi Residual (Level) - {nama_terbaik}")
axes[0].set_xlabel("Residual (Rupiah)"); axes[0].set_ylabel("Frekuensi")
axes[1].scatter(level_terbaik, residual, alpha=0.3, s=10, color="purple")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title("Residual vs Nilai Prediksi")
axes[1].set_xlabel("Prediksi (Rupiah)"); axes[1].set_ylabel("Residual (Rupiah)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "08_distribusi_residual.png"), bbox_inches="tight")
plt.show()
print(f"Rata-rata residual: {residual.mean():.2f} | Std residual: {residual.std():.2f}")

In [ ]:
# Feature importance RF & XGBoost
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
pd.Series(rf_model.feature_importances_, index=FEATURE_COLS).sort_values().plot(
    kind="barh", ax=axes[0], color="#55A868")
axes[0].set_title("Feature Importance - Random Forest")
pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values().plot(
    kind="barh", ax=axes[1], color="#4C72B0")
axes[1].set_title("Feature Importance - XGBoost")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "09_feature_importance.png"), bbox_inches="tight")
plt.show()

**Interpretasi evaluasi (untuk keputusan bisnis):**
- **R^2 level tinggi untuk semua model & baseline** karena begitu level direkonstruksi dari harga
  kemarin, sebagian besar variansi otomatis terjelaskan. Jadi **R^2 bukan pembeda yang berarti** di sini.
- **Pada RMSE/MAE, model juara mengalahkan baseline naif namun hanya tipis** (setelah koreksi
  data). Perbedaannya kecil karena memprediksi **angka persis** kurs harian memang sangat sulit
  (pasar mendekati efisien) - keunggulan tipis ini tetap berarti tapi bukan keunggulan utama.
- **Akurasi arah adalah metrik penentu.** Model juara menebak arah jauh di atas 50% (tebakan koin) -
  inilah keunggulan nyata yang bisa dimanfaatkan untuk *timing* keputusan valas.
- **Feature importance** menunjukkan fitur momentum jangka pendek (lag return) & volatilitas
  (rolling std) paling informatif - konsisten dengan literatur *short-term momentum* &
  *volatility clustering*.

## 8. Forecasting 30 Hari ke Depan

Proyeksi USD/IDR 30 hari kerja ke depan dengan model juara secara **iteratif** di ruang return,
lalu direkonstruksi ke level.

**Asumsi:** perubahan indikator makro ke depan diasumsikan **nol** (makro relatif stabil), sehingga
forecast bergantung pada dinamika return USD/IDR. **Confidence band melebar** mengikuti horizon
(`sigma * sqrt(h)`) - representasi ketidakpastian yang jujur (makin jauh, makin tak pasti).

In [ ]:
HORIZON = 30
tanggal_terakhir = d_model.index[-1]
tanggal_masa_depan = pd.bdate_range(start=tanggal_terakhir + pd.tseries.offsets.BDay(1), periods=HORIZON)

ret_hist = d_model["ret"].tolist()
harga_terakhir = float(d_model["USDIDR"].iloc[-1])
rate_diff_terakhir = float(d_model["rate_diff"].iloc[-1])
sigma_ret = float(np.std(actual_ret - RET_TERBAIK))  # sigma error one-step (ruang return)

harga_kerja = harga_terakhir
prediksi_level, batas_bawah, batas_atas, prediksi_ret = [], [], [], []

for h, tgl in enumerate(tanggal_masa_depan, start=1):
    fitur = {
        "ret_lag1": ret_hist[-1], "ret_lag2": ret_hist[-2], "ret_lag3": ret_hist[-3],
        "ret_lag5": ret_hist[-5], "ret_lag10": ret_hist[-10],
        "ret_roll7_mean": float(np.mean(ret_hist[-7:])),
        "ret_roll7_std": float(np.std(ret_hist[-7:])),
        "ret_roll30_mean": float(np.mean(ret_hist[-30:])),
        "ret_roll30_std": float(np.std(ret_hist[-30:])),
        "OIL_chg_lag1": 0.0, "GOLD_chg_lag1": 0.0, "SP500_chg_lag1": 0.0,
        "IHSG_chg_lag1": 0.0, "VIX_chg_lag1": 0.0, "CPI_chg_lag1": 0.0,
        "rate_diff_lag1": rate_diff_terakhir, "rate_diff_chg_lag1": 0.0,
        "day_of_week": tgl.dayofweek, "month": tgl.month,
    }
    Xf = pd.DataFrame([fitur])[FEATURE_COLS]
    Xf_scaled = scaler.transform(Xf)
    r_hat = float(MODEL_TERBAIK.predict(Xf_scaled)[0])
    harga_kerja = harga_kerja * np.exp(r_hat)
    prediksi_ret.append(r_hat); prediksi_level.append(harga_kerja)
    lebar = sigma_ret * np.sqrt(h)
    batas_bawah.append(harga_kerja * np.exp(-lebar))
    batas_atas.append(harga_kerja * np.exp(+lebar))
    ret_hist.append(r_hat)

df_forecast = pd.DataFrame({
    "Tanggal": tanggal_masa_depan,
    "Prediksi_Return": np.round(prediksi_ret, 6),
    "Prediksi_USDIDR": np.round(prediksi_level, 2),
    "Batas_Bawah": np.round(batas_bawah, 2),
    "Batas_Atas": np.round(batas_atas, 2),
}).set_index("Tanggal")
print("=== FORECAST 30 HARI (model juara:", nama_terbaik, ") ===")
display(df_forecast)
df_forecast.to_csv(os.path.join(OUTPUT_DIR, "forecast_30_hari.csv"))

In [ ]:
plt.figure(figsize=(15, 7))
hist_tail = d_model["USDIDR"].iloc[-90:]
plt.plot(hist_tail.index, hist_tail.values, label="Historis (90 hari terakhir)", color="black")
plt.plot(df_forecast.index, df_forecast["Prediksi_USDIDR"], label=f"Forecast ({nama_terbaik})",
         color="orange", marker="o", markersize=3)
plt.fill_between(df_forecast.index, df_forecast["Batas_Bawah"], df_forecast["Batas_Atas"],
                 color="orange", alpha=0.2, label="Confidence band (+/- 1 sigma*sqrt(h))")
plt.axvline(tanggal_terakhir, color="gray", linestyle="--", alpha=0.7, label="Akhir data historis")
plt.title("Forecasting USD/IDR 30 Hari ke Depan")
plt.xlabel("Tanggal"); plt.ylabel("USD/IDR"); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "10_forecasting_30_hari.png"), bbox_inches="tight")
plt.show()

**Interpretasi forecasting:** karena model bekerja pada return dan perubahan makro diasumsikan
nol, lintasan forecast mengikuti momentum return terakhir secara halus. **Band yang melebar**
mencerminkan ketidakpastian yang bertambah seiring horizon. Forecast ini adalah **alat bantu
perencanaan**, bukan kepastian.

## 9. Insight dan Rekomendasi Bisnis

In [ ]:
# Periode anomali pada level
plt.figure(figsize=(15, 6))
plt.plot(df_clean.index, df_clean["USDIDR"], color="darkred", linewidth=1)
plt.axvspan(pd.Timestamp("2020-02-01"), pd.Timestamp("2020-06-30"), color="blue", alpha=0.15,
            label="Guncangan COVID-19 (2020)")
plt.axvspan(pd.Timestamp("2022-03-01"), pd.Timestamp("2023-07-31"), color="green", alpha=0.15,
            label="Siklus kenaikan Fed rate (2022-2023)")
plt.title("Periode Anomali pada Nilai Tukar USD/IDR")
plt.xlabel("Tahun"); plt.ylabel("USD/IDR"); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "11_periode_anomali.png"), bbox_inches="tight")
plt.show()

### 9.1 Temuan Utama
- **Nilai tukar mendekati efisien dalam jangka pendek.** Model hanya unggul tipis atas baseline
  naif pada RMSE, menegaskan return harian USD/IDR sangat mendekati *random walk*. Temuan akademik valid.
- **Namun arah pergerakan dapat diprediksi sebagian** (akurasi arah model juara > 50%) - inilah nilai
  praktis utama sistem ini untuk *timing* keputusan valas.
- **Momentum & volatilitas jangka pendek** (lag return, rolling std) menjadi fitur paling informatif.
- **Selisih suku bunga & perubahan komoditas/volatilitas** berperan sebagai konteks makro, terutama
  saat rezim pasar berubah.

### 9.2 Periode Anomali
- **COVID-19 (Mar-Apr 2020):** lonjakan tajam akibat *flight to safety*.
- **Siklus kenaikan Fed (2022-2023):** penguatan Dolar global menekan Rupiah secara persisten.

### 9.3 Lima Rekomendasi Keputusan Bisnis
1. **Hedging berbasis risiko, bukan titik prediksi.** Gunakan **lebar confidence band** sebagai
   dasar ukuran lindung nilai: makin lebar band, makin besar porsi yang di-hedge.
2. **Jadikan baseline naif sebagai acuan operasional** untuk perencanaan kas sangat pendek.
3. **Sistem peringatan dini berbasis volatilitas:** pantau lonjakan rolling std return & `VIX`.
4. **Manfaatkan sinyal arah secara hati-hati:** bila akurasi arah konsisten di atas ~55% pada
   *backtesting* berkelanjutan, sinyal dipakai sebagai input tambahan (bukan tunggal) untuk *timing*.
5. **Retraining & backtesting berkala** (mis. bulanan), selalu dievaluasi relatif terhadap baseline.

### 9.4 Analisis Risiko Model
- **Kapan keliru:** guncangan eksogen (pandemi, geopolitik, intervensi bank sentral), perubahan
  rezim moneter, horizon panjang (error iteratif terakumulasi).
- **Mitigasi:** perlakukan model sebagai pendukung keputusan, gunakan band ketidakpastian,
  kombinasikan analisis fundamental, terapkan disiplin manajemen risiko, dan backtesting rutin.

## 10. Simpan Model (untuk Aplikasi)

Menyimpan model juara, scaler, dan metadata. **Penting:** karena target adalah **return**, proses
inferensi di aplikasi WAJIB merekonstruksi level dengan `USDIDR_t = USDIDR_{t-1} * exp(return_hat)`.

In [ ]:
path_model  = os.path.join(OUTPUT_DIR, "model_terbaik_usdidr_return.pkl")
path_scaler = os.path.join(OUTPUT_DIR, "scaler_minmax.pkl")
path_meta   = os.path.join(OUTPUT_DIR, "metadata_fitur.pkl")

joblib.dump(MODEL_TERBAIK, path_model)
joblib.dump(scaler, path_scaler)
joblib.dump({
    "nama_model": nama_terbaik,
    "feature_cols": FEATURE_COLS,
    "target": "ret (log-return USDIDR)",
    "kriteria_pemilihan": "directional_accuracy",
    "metrik": tabel_eval.loc[nama_terbaik].to_dict(),
    "harga_terakhir": float(d_model["USDIDR"].iloc[-1]),
    "tanggal_terakhir": str(d_model.index[-1].date()),
    "rate_diff_terakhir": float(d_model["rate_diff"].iloc[-1]),
    "ret_hist_terakhir": d_model["ret"].tolist()[-30:],
    "sigma_ret": sigma_ret,
    "cara_inferensi": "USDIDR_t = USDIDR_(t-1) * exp(prediksi_return)",
}, path_meta)
print("Artefak tersimpan di folder:", OUTPUT_DIR)
print(" -", path_model); print(" -", path_scaler); print(" -", path_meta)
print("Model juara:", nama_terbaik)

In [ ]:
# Contoh memuat ulang & inferensi (simulasi aplikasi)
model_loaded  = joblib.load(path_model)
scaler_loaded = joblib.load(path_scaler)
meta_loaded   = joblib.load(path_meta)

baris = test.iloc[[0]]
Xc = baris[meta_loaded["feature_cols"]]
ret_hat = float(model_loaded.predict(scaler_loaded.transform(Xc))[0])
harga_kemarin = float(baris["USDIDR_prev"].iloc[0])
prediksi_level = harga_kemarin * np.exp(ret_hat)
arah = "NAIK" if ret_hat > 0 else "TURUN"

print("Model dimuat ulang:", meta_loaded["nama_model"])
print(f"Prediksi return : {ret_hat:.6f}  -> arah {arah}")
print(f"Harga kemarin   : {harga_kemarin:.2f}")
print(f"Prediksi level  : {prediksi_level:.2f}")
print(f"Aktual level    : {float(baris['USDIDR'].iloc[0]):.2f}")
print("\nSiap diintegrasikan ke aplikasi Flask (siap deploy di cPanel).")

## Kesimpulan Akhir

Sistem ini memodelkan **return harian (stasioner)** alih-alih level harga, dengan **baseline naif**
sebagai tolok ukur, **tiga model** sebagai pembanding (linear, bagging, boosting), dan **akurasi
arah** sebagai kriteria pemilihan juara.

**Alur ringkas:**
1. **EDA** - deret non-stasioner, ada periode anomali (COVID-19, siklus Fed), dan **2 kesalahan input
   USDIDR yang dikoreksi**.
2. **Preprocessing** - target return + fitur stasioner bebas *look-ahead bias*.
3. **Tiga model** dilatih pada return.
4. **Evaluasi jujur** vs baseline; juara dipilih dari **akurasi arah**.
5. **Forecasting 30 hari** dengan band ketidakpastian yang melebar.
6. **Insight bisnis** berbasis risiko.
7. **Model disimpan** lengkap dengan metadata untuk aplikasi.

**Catatan penutup yang jujur:** pada nilai tukar harian, keunggulan atas baseline naif untuk **angka
persis** hanya tipis karena pasar mendekati efisien. Nilai utama proyek ini adalah **kerangka
analisis yang benar secara metodologis, jujur dalam evaluasi, mampu memprediksi ARAH jelas lebih baik
dari tebakan acak (~65%), dan berguna untuk manajemen risiko** - bukan sekadar R^2 tinggi yang menyesatkan.

---
*Seluruh plot tersimpan di folder `output/` (.png) dan artefak model dalam format `.pkl`.*